# 01 · De CSV a submission en 10 minuts

Objectiu: tenir **una puntuació al leaderboard en menys de 2 hores** des de l'inici. Després s'itera sobre aquesta base.

> Ve configurada amb Schneider 2025 (classificació). Per a regressió, canvieu `TASK` i `METRIC`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from dreamteam import data, metrics, models, submission, validation

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

In [ ]:
# ── CONFIG ──────────────────────────────────────────────
TRAIN_PATH = ROOT / "data/past/2025_schneider/dataset.csv"
TEST_PATH  = None            # el test.csv oficial; si és None, fem un holdout del train
TARGET  = "target_variable"
ID_COL  = "id"
TASK    = "binary"           # "binary" | "regression" | "quantile" | "tweedie"
METRIC  = metrics.auc        # metrics.msle, metrics.rmse, metrics.asymmetric_error…
PROBA   = TASK == "binary"
# ────────────────────────────────────────────────────────

In [ ]:
train = data.load(TRAIN_PATH)
if TEST_PATH:
    test = data.load(TEST_PATH)
else:
    train, test = train_test_split(train, test_size=0.2, random_state=42,
                                   stratify=train[TARGET] if PROBA else None)

X, y = data.split_xy(train, TARGET, drop=[ID_COL])
X = data.to_category(X)
X_test = data.to_category(test[X.columns])
for col in X.select_dtypes("category"):
    X_test[col] = pd.Categorical(X_test[col], categories=X[col].cat.categories)
print(X.shape, X_test.shape)

## Baseline 0: el més tonto possible
Si el model no supera això, alguna cosa va malament.

In [ ]:
dummy = np.full(len(y), y.mean())
print(f"Baseline constant: {METRIC(y, dummy):.5f}")

## Baseline 1: LightGBM amb CV
Trieu el split amb cap: `kfold` / `group_kfold` / `time_split` (vegeu `src/dreamteam/validation.py`).

In [ ]:
# Hi ha files duplicades? Si n'hi ha, un K-Fold aleatori infla la CV (Schneider 2025: 77 %!)
dup = validation.duplicate_report(X)
groups = validation.duplicate_groups(X) if dup > 0.05 else None

model = models.lgbm_baseline(TASK, n_estimators=500, learning_rate=0.05)
cv = validation.group_kfold(5) if groups is not None else validation.kfold(5, stratified=PROBA)
oof, scores, fold_models = validation.cross_validate(model, X, y, cv=cv, metric=METRIC,
                                                     groups=groups, proba=PROBA)

In [ ]:
# Importància ràpida (guany): on mirar per fer feature engineering
imp = pd.Series(fold_models[0].booster_.feature_importance("gain"), index=X.columns)
imp.sort_values().tail(15).plot.barh(figsize=(7, 5), title="Importància (gain)");

## Ajust a la mètrica
Classificació amb F1 → llindar òptim. Pèrdua asimètrica → factor d'escala. **Sempre sobre OOF.**

In [ ]:
if PROBA:
    thr, f1 = metrics.best_f1_threshold(y, oof)
    print(f"Llindar òptim F1: {thr:.2f} (F1 OOF = {f1:.3f})")
else:
    k, s = metrics.best_scale_factor(y, oof)
    print(f"Factor òptim per a la pèrdua asimètrica: ×{k:.2f} (error {s:.4f})")

## Submission

In [ ]:
preds = validation.predict_folds(fold_models, X_test, proba=PROBA)
if not TEST_PATH:
    # Ull: si hi ha duplicats, aquest holdout aleatori també surt inflat. Mireu la CV agrupada.
    print(f"Holdout: {METRIC(test[TARGET], preds):.5f}")
path = submission.make_submission(test[ID_COL], preds, id_col=ID_COL, target_col=TARGET,
                                  out_dir=ROOT / "submissions", name="baseline")

## Següents passos
- [ ] Apuntar el resultat a `experiments.md`
- [ ] Feature engineering (vegeu `docs/04_playbook_ml.md`)
- [ ] CatBoost i ensemble
- [ ] Optuna, només amb les features congelades